# 중간 프로젝트 - Wildberries 크롤러

## 2. 리뷰 크롤링

### 상품 1건당(총 10건) 리뷰 200개씩 수집 후 엑셀 파일 생성

> **리뷰 엑셀 파일 컬럼**: Review_ID | Rating | Review_Text | Sorter
>
> 각 행은 하나의 리뷰 데이터

#### 리뷰 엑셀 파일의 시트 구성

> Sheet1: 1위 상품의 리뷰 10건 (별점 높은 순 5건 / 낮은 순 5건)  
> Sheet2: 2위 상품의 리뷰 10건  
> ...  
> Sheet10: 10위 상품의 리뷰 10건

In [1]:
# Step 1. 필요한 라이브러리 로딩
# !pip install undetected-chromedriver
# !pip install --upgrade undetected-chromedriver
import pandas as pd
import re
import openpyxl
import os, time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import random

In [2]:
# Step 2. 생성한 엑셀 파일 정보 확인 & 리뷰 URL 컬럼 불러오기
df = pd.read_excel(r"C:\Users\itwill\workspace\Project\중간 프로젝트\260428_Ru_Wildberries_Cream\260428_Ru_Wildberries_Cream.xlsx")
print(df.columns)
print(df.info())

rev_page = df['Review_URL'].tolist()  # 각 상품의 리뷰 페이지 URL 리스트 → for 반복문에 사용
print(f"리뷰 페이지 리스트 길이: {len(rev_page)}개")
rev_page

Index(['Ranking', 'Image', 'Product_Name', 'Brand', 'Price(₽)', 'Rating',
       'Review_Cnt', 'Ingredients', 'Benefits', 'Skin_Type', 'URL',
       'Review_URL'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ranking       10 non-null     int64  
 1   Image         0 non-null      float64
 2   Product_Name  10 non-null     str    
 3   Brand         10 non-null     str    
 4   Price(₽)      10 non-null     int64  
 5   Rating        10 non-null     float64
 6   Review_Cnt    10 non-null     int64  
 7   Ingredients   10 non-null     str    
 8   Benefits      10 non-null     str    
 9   Skin_Type     10 non-null     str    
 10  URL           10 non-null     str    
 11  Review_URL    10 non-null     str    
dtypes: float64(2), int64(3), str(7)
memory usage: 1.1 KB
None
리뷰 페이지 리스트 길이: 10개


['https://www.wildberries.ru/catalog/493818806/feedbacks?size=687154703',
 'https://www.wildberries.ru/catalog/624803241/feedbacks?size=847037607',
 'https://www.wildberries.ru/catalog/172282016/feedbacks?size=285878653',
 'https://www.wildberries.ru/catalog/90002926/feedbacks?size=145011801',
 'https://www.wildberries.ru/catalog/461393225/feedbacks?size=649393160',
 'https://www.wildberries.ru/catalog/258293575/feedbacks?size=401925959',
 'https://www.wildberries.ru/catalog/378889076/feedbacks?size=551527551',
 'https://www.wildberries.ru/catalog/260889862/feedbacks?size=405408531',
 'https://www.wildberries.ru/catalog/863074092/feedbacks?size=1299521121',
 'https://www.wildberries.ru/catalog/8104661/feedbacks?size=27659229']

In [3]:
# Step 3. undetected-chromedriver 실행
driver = uc.Chrome(version_main=147)

driver.get('https://www.wildberries.ru/catalog/krasota/uhod-za-kozhey/uhod-za-litsom')
WebDriverWait(driver, 60).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'span.goods-count'))
)  # 상품 수 요소가 로딩될 때까지 최대 60초 대기
driver.maximize_window()
time.sleep(random.uniform(1, 3))

In [4]:
# Step 4. 유틸 함수 정의

def scroll_to_load(driver, target=100):
    """
    무한 스크롤(lazy loading) 구조에서 리뷰가 target개 이상 로딩될 때까지
    페이지 하단으로 반복 스크롤한다.
    - 새 리뷰가 더 이상 로딩되지 않으면(3회 연속 동일) 조기 종료
    - 반환값: 실제 로딩된 리뷰 요소 리스트
    """
    CSS_REVIEW = "ul.comments__list > li"
    no_change  = 0

    while True:
        els = driver.find_elements(By.CSS_SELECTOR, CSS_REVIEW)
        cur = len(els)

        if cur >= target:
            break

        # 마지막 리뷰 요소까지 스크롤
        if els:
            driver.execute_script(
                "arguments[0].scrollIntoView({behavior: 'smooth', block: 'end'});", els[-1]
            )
        else:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        time.sleep(random.uniform(1.5, 2.5))   # 로딩 대기

        new_els = driver.find_elements(By.CSS_SELECTOR, CSS_REVIEW)
        if len(new_els) == cur:   # 스크롤 후에도 개수 변화 없음
            no_change += 1
            if no_change >= 3:    # 3회 연속 변화 없으면 더 로딩할 리뷰 없음
                print(f"    더 이상 로딩할 리뷰 없음 (현재 {cur}건)")
                break
        else:
            no_change = 0         # 새 리뷰 로딩되면 카운터 초기화
            print(f"    스크롤 로딩 중... {cur} → {len(new_els)}건")

    return driver.find_elements(By.CSS_SELECTOR, CSS_REVIEW)


def extract_review_text(soup_obj, review_el):
    """BeautifulSoup 객체와 개별 리뷰 Selenium 요소에서 리뷰 본문을 추출"""
    try:
        # 개별 리뷰 요소 기준으로 파싱 (페이지 전체 soup 대신 해당 리뷰 HTML만 파싱)
        review_soup = BeautifulSoup(review_el.get_attribute('outerHTML'), 'html.parser')

        # 장점 (Достоинства)
        pro = review_soup.select_one("span.feedback__text--item-pro")
        pro_text = pro.get_text(separator=" ", strip=True) if pro else ""

        # 단점 (Недостатки)
        con = review_soup.select_one("span.feedback__text--item-con")
        con_text = con.get_text(separator=" ", strip=True) if con else ""

        # 일반 댓글 (Комментарий)
        comment_el = review_soup.select_one(
            "span.feedback__text--item:not(.feedback__text--item-pro):not(.feedback__text--item-con)"
        )
        comment_text = comment_el.get_text(separator=" ", strip=True) if comment_el else ""

        # 태그형 키워드 리스트 (Плюсы / Минусы товара)
        bables_title_el = review_soup.select_one("span.feedbacks-bables__title")
        bables_items    = review_soup.select("ul.feedbacks-bables__list li.feedbacks-bables__item")
        bables_text = ""
        if bables_title_el and bables_items:
            title      = bables_title_el.get_text(strip=True)
            items_str  = ", ".join([li.get_text(strip=True) for li in bables_items])
            bables_text = f"{title}: {items_str}"

        all_contents = [pro_text, con_text, comment_text, bables_text]
        return "\n".join([t for t in all_contents if t]) or 'Null'

    except Exception:
        return 'Null'


def extract_rating(review_el):
    """개별 리뷰 Selenium 요소에서 별점을 추출"""
    try:
        review_soup = BeautifulSoup(review_el.get_attribute('outerHTML'), 'html.parser')
        star_tag    = review_soup.select_one('span.stars-line.feedback__rating')
        if star_tag:
            star_cls = star_tag.get('class', [])
            star_val = next((c for c in star_cls if re.match(r'^star\d+$', c)), None)
            return float(star_val.replace('star', '')) if star_val else 0.0
    except Exception:
        pass
    return 0.0

In [5]:
# Step 5. 저장 경로 설정
s = time.strftime('%y%m%d', time.localtime())   # 현재 날짜 'YYMMDD' 형식

f_dir = input("저장 경로 (Enter: 기본 경로 사용): ")
if f_dir == '':
    f_dir = r"C:\Users\itwill\workspace\Project\중간 프로젝트"
elif f_dir == '1':
    f_dir = r"D:\workspace"

base_name = f'{s}_Ru_Wildberries_Cream'
folder    = os.path.join(f_dir, base_name)
os.chdir(folder)

fr_name = os.path.join(folder, base_name + '_Reviews.xlsx')
print(f"리뷰 파일명: {fr_name}")

리뷰 파일명: C:\Users\itwill\workspace\Project\중간 프로젝트\260428_Ru_Wildberries_Cream\260428_Ru_Wildberries_Cream_Reviews.xlsx


In [ ]:
# Step 6. 리뷰 데이터 수집 & 엑셀 저장
# - 상품당 별점 높은 순 100건 + 낮은 순 100건 = 200건
# - 상품 1개 = 엑셀 시트 1개 (rank_01 ~ rank_10)

wb_rev   = openpyxl.Workbook()
wb_rev.remove(wb_rev.active)   # 기본 빈 시트 제거
ranking  = df['Ranking'].tolist()

# ── 정렬 버튼 셀렉터 상수 ─────────────────────────────────────────────────────
# HTML 구조: <li class="sorting__item"> <a type="button" aria-label="Оценке" ...>
# 현재 선택된 버튼: .sorting__selected
# 방향 표시: span.sorting__decor (기본 ↓높은 순) / span.sorting__decor--up (↑낮은 순)
CSS_SORT_BTN  = "a[type='button'][aria-label='Оценке']"   # 별점 정렬 버튼 (공통)
CSS_SORT_DESC = f"{CSS_SORT_BTN}:not(.sorting__selected)" # 현재 미선택 = 클릭 전 대상
CSS_IS_UP     = f"{CSS_SORT_BTN} span.sorting__decor--up" # ↑(낮은 순) 방향 아이콘 존재 여부
# ──────────────────────────────────────────────────────────────────────────────


def click_sort_btn(driver, wait_css, label):
    """
    정렬 버튼을 클릭하고, 클릭 성공 여부를 wait_css 요소의 등장으로 검증한다.
    wait_css : 클릭 후 DOM에 나타나야 할 요소의 CSS 셀렉터 (정렬 적용 확인용)
    label    : 로그 출력용 텍스트
    반환값   : 성공 True / 실패 False
    """
    try:
        # 1) 버튼 요소 탐색 및 화면 중앙 스크롤
        btn = WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, CSS_SORT_BTN))
        )
        driver.execute_script(
            "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", btn
        )
        time.sleep(0.5)   # 스크롤 완료 대기

        # 2) 클릭
        btn.click()
        time.sleep(random.uniform(1, 2))

        # 3) 클릭 결과 검증: wait_css 요소가 DOM에 나타나는지 확인
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, wait_css))
        )
        print(f"  ✅ {label} 정렬 적용 확인")
        return True

    except Exception as e:
        print(f"  ❌ {label} 정렬 버튼 클릭 실패: {e}")
        return False


for i, rev_url in enumerate(rev_page):
    rank        = ranking[i]
    sheet_label = f"rank_{rank:02d}"
    sheet_label = re.sub(r'[\\/:*?"<>|]', '', sheet_label)[:10]
    ws_rev      = wb_rev.create_sheet(title=sheet_label)
    ws_rev.append(['Review_ID', 'Rating', 'Review_Text', 'Sorter'])   # 헤더

    print(f"\n[{rank}위] 리뷰 페이지: {rev_url}")

    if rev_url == 'Null' or pd.isna(rev_url):
        print("  → 리뷰 URL 없음, 시트만 생성")
        continue

    # ── 리뷰 페이지 접속 ──────────────────────────────────────────────────────
    try:
        driver.get(rev_url)
        WebDriverWait(driver, 120).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "ul.comments__list > li"))
        )
        print(f"  {rank}위 상품 리뷰 페이지 접속 완료")
    except Exception as e:
        print(f"  → 페이지 접속 실패: {e}")
        continue

    # ── ① 별점 높은 순 정렬 클릭 & 검증 ─────────────────────────────────────
    # 페이지 기본값이 이미 높은 순(↓)일 수 있으므로, 현재 상태를 먼저 확인
    already_desc = bool(driver.find_elements(By.CSS_SELECTOR, CSS_SORT_BTN + ".sorting__selected"))
    is_up        = bool(driver.find_elements(By.CSS_SELECTOR, CSS_IS_UP))

    if already_desc and not is_up:
        # 이미 높은 순(↓) 선택 상태 → 클릭 불필요
        print("  ✅ 이미 별점 높은 순 정렬 상태")
    else:
        # 별점 정렬 버튼이 미선택이거나 낮은 순(↑) 상태 → 클릭하여 높은 순으로 전환
        # 검증 기준: 클릭 후 .sorting__selected 가 되고 span.sorting__decor--up 이 사라짐
        ok = click_sort_btn(
            driver,
            wait_css = f"{CSS_SORT_BTN}.sorting__selected",
            label    = "별점 높은 순"
        )
        if not ok:
            print("  → 높은 순 정렬 실패, 현재 정렬 상태로 수집 진행")

    # 정렬 후 100건 로딩될 때까지 스크롤
    print("  높은 순 리뷰 스크롤 로딩 시작...")
    review_els = scroll_to_load(driver, target=100)

    for count, review_el in enumerate(review_els[:100], start=1):   # 최대 100건
        print(f"  높은 순 {count}번째 리뷰 수집 중...")
        driver.execute_script(
            "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", review_el
        )
        time.sleep(random.uniform(0.3, 0.8))

        rev_id     = count
        rev_rating = extract_rating(review_el)
        rev_txt    = extract_review_text(None, review_el)
        sorter     = f"높은 순 {count}번째"

        print(f"    Review_ID: {rev_id} | Rating: {rev_rating} | Sorter: {sorter}")
        ws_rev.append([rev_id, rev_rating, rev_txt, sorter])

    # ── ② 별점 낮은 순 정렬 클릭 & 검증 ─────────────────────────────────────
    # 검증 기준: 클릭 후 span.sorting__decor--up(↑) 아이콘이 DOM에 등장
    ok = click_sort_btn(
        driver,
        wait_css = CSS_IS_UP,
        label    = "별점 낮은 순"
    )
    if not ok:
        print("  → 낮은 순 정렬 실패, 현재 정렬 상태로 수집 진행")

    # 정렬 후 100건 로딩될 때까지 스크롤
    print("  낮은 순 리뷰 스크롤 로딩 시작...")
    review_els = scroll_to_load(driver, target=100)

    for count, review_el in enumerate(review_els[:100], start=1):   # 최대 100건
        print(f"  낮은 순 {count}번째 리뷰 수집 중...")
        driver.execute_script(
            "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", review_el
        )
        time.sleep(random.uniform(0.3, 0.8))

        rev_id     = count + 100   # 101번부터 시작 (높은 순 이어서)
        rev_rating = extract_rating(review_el)
        rev_txt    = extract_review_text(None, review_el)
        sorter     = f"낮은 순 {count}번째"

        print(f"    Review_ID: {rev_id} | Rating: {rev_rating} | Sorter: {sorter}")
        ws_rev.append([rev_id, rev_rating, rev_txt, sorter])

    print(f"  [{rank}위] 총 200건 수집 완료")
    time.sleep(random.uniform(1, 2))   # 상품 간 딜레이

# ── 저장 ──────────────────────────────────────────────────────────────────────
wb_rev.save(fr_name)
print("\n" + "=" * 60)
print(f"리뷰 엑셀 저장 완료: {fr_name}")
print(f"총 {len(rev_page)}개 상품, 시트 수: {len(wb_rev.sheetnames)}")

In [ ]:
# Step 7. 드라이버 종료
driver.quit()
print("드라이버 종료 완료")